<a href="https://colab.research.google.com/github/Luis098765/ml-with-aurelien-geron/blob/main/project/exercises/ch03/Chapter_3_Exercises_1_%26_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3 - Exercises 1 & 2

#### 1. Try to build a classifier for the MNIST dataset that achieves over 97% accuracy on the test set. Hint: the KNeighborsClassifier works quite well for this task; you just need to find good hyperparameter values (try a grid search on the weights and n_neighbors hyperparameters).
#### P.s: sgd (not scaled) = 0.874
#### R.: knn(weights='distance', n_neighbors=4) (not scaled) = 0.9714

#### 2. Write a function that can shift an MNIST image in any direction (left, right, up,or down) by one pixel. Then, for each image in the training set, create four shifted copies (one per direction) and add them to the training set. Finally, train yourbest model on this expanded training set and measure its accuracy on the test set. You should observe that your model performs even better now! This technique of artificially growing the training set is called data augmentation or training set expansion.
#### R.: o desempenho após o processo foi de 0.9763, mas o tempo de treino aumentou consideravelmente (37s para 3min).

## Imports

In [1]:
from sklearn.datasets import fetch_openml
from scipy.ndimage import shift

import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import cross_val_score, GridSearchCV

from sklearn.pipeline import Pipeline

## Data obtainment & split

In [2]:
mnist = fetch_openml('mnist_784', version=1)
mnist.keys()

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])

In [3]:
def split_mnist(mnist):
  X, y = mnist["data"].to_numpy(), mnist["target"].to_numpy()
  return X[:60000], X[60000:], y[:60000], y[60000:]

In [4]:
X_train, X_test, y_train, y_test = split_mnist(mnist)

## Model Training

In [5]:
pipe_no_scaler = Pipeline([
    ("knn", KNeighborsClassifier())
])

pipe_scaler = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = [
    {'knn__weights': ['uniform', 'distance'], 'knn__n_neighbors': [3, 4, 5, 6]}
]

In [6]:
grid_no_scaler = GridSearchCV(pipe_no_scaler, param_grid, cv=3, scoring='accuracy', n_jobs=2, return_train_score=False)

#grid_no_scaler.fit(X_train, y_train)

In [7]:
grid_scaler = GridSearchCV(pipe_scaler, param_grid, cv=3, scoring='accuracy', n_jobs=2, return_train_score=False)

#grid_scaler.fit(X_train, y_train)

In [8]:
def print_results():
  print("Melhores modelos e desempenhos:")
  print("-"*20)
  print("Sem normalização:")
  print("Melhor modelo:", grid_no_scaler.best_estimator_)
  print("Desempenho:", grid_no_scaler.best_score_)
  print("-"*20)
  print("Com normalização:")
  print("Melhor modelo:", grid_scaler.best_estimator_)
  print("Desempenho:", grid_scaler.best_score_)
  print("-"*20)

In [9]:
#print_results()
'''
Melhores modelos e desempenhos:
--------------------
Sem normalização:
Melhor modelo: Pipeline(steps=[('knn',
                 KNeighborsClassifier(n_neighbors=4, weights='distance'))])
Desempenho: 0.9703500000000002
--------------------
Com normalização:
Melhor modelo: Pipeline(steps=[('scaler', StandardScaler()),
                ('knn',
                 KNeighborsClassifier(n_neighbors=4, weights='distance'))])
Desempenho: 0.9439333333333334
--------------------
'''

"\nMelhores modelos e desempenhos:\n--------------------\nSem normalização:\nMelhor modelo: Pipeline(steps=[('knn',\n                 KNeighborsClassifier(n_neighbors=4, weights='distance'))])\nDesempenho: 0.9703500000000002\n--------------------\nCom normalização:\nMelhor modelo: Pipeline(steps=[('scaler', StandardScaler()),\n                ('knn',\n                 KNeighborsClassifier(n_neighbors=4, weights='distance'))])\nDesempenho: 0.9439333333333334\n--------------------\n"

In [10]:
knn_clf = KNeighborsClassifier(weights='distance', n_neighbors=4)

knn_clf.fit(X_train, y_train)

knn_clf.score(X_test, y_test)

0.9714

## After data augmentation

In [13]:
def shifted_images(images):
  images = images.reshape(-1, 28, 28)

  shift_1 = shift(images, [0, 0, 1], cval=0)
  shift_2 = shift(images, [0, 0, -1], cval=0)
  shift_3 = shift(images, [0, 1, 0], cval=0)
  shift_4 = shift(images, [0, -1, 0], cval=0)

  return np.r_[images, shift_1, shift_2, shift_3, shift_4].reshape(-1, 784)

In [14]:
X_train_aug = shifted_images(X_train)
y_train_aug = np.tile(y_train, 5)

print(X_train_aug.shape, y_train_aug.shape)

(300000, 784) (300000,)
(300000, 784) (300000,)


In [15]:
knn_clf = KNeighborsClassifier(weights='distance', n_neighbors=4)

knn_clf.fit(X_train_aug, y_train_aug)

knn_clf.score(X_test, y_test)

0.9763